# Notebook 01 — Core RL Concepts

**Who this is for:** Makers and hobbyists who are curious about reinforcement
learning but have never coded an RL agent before.  No prior ML experience
is assumed.

**What you will learn:**

| Concept | Short definition |
|---------|-----------------|
| Agent | The learner — decides what to do next |
| Environment | The world the agent acts in |
| State / Observation | What the agent can see |
| Action | What the agent can do |
| Reward | A scalar signal that tells the agent how well it did |
| Policy | A mapping from observations to actions |
| Episode | One complete trial, from reset to termination |
| Exploration vs Exploitation | When to try something new vs stick with what works |

We build intuition with the simplest possible RL problem — a **multi-armed
bandit** — and then connect each idea to the Buddy Jr robot arm.

---

## 1. The RL Loop

Every RL problem has the same shape:

```
         observation
Agent  <------------- Environment
  |                       ^
  |  action               |  reward
  +----------------------->
```

1. The **environment** gives the agent an **observation** (a description of the
   current state of the world).
2. The **agent** picks an **action**.
3. The environment transitions to a new state and returns a **reward** and the
   next observation.
4. Repeat until the episode ends.

For Buddy Jr the observation is a 17-number vector describing joint angles,
the camera-tip position, and the target position.  The action is a small
per-joint velocity command.  The reward is based on how much closer the arm
moved to the target.

## 2. The Simplest RL Problem: a Bandit

A **multi-armed bandit** has no state and no sequence — just one repeated
choice.  Each arm has a hidden mean reward; the agent must discover which arm
is best by trying them.

We model Buddy Jr's `base_yaw` joint as a 5-armed bandit.  The joint can snap
to one of five angles: −90°, −45°, 0°, +45°, +90°.  One angle has a higher
hidden reward (the robot faces a target there); the others do not.

In [ ]:
import numpy as np
import math

# ---------- constants (mirrors experiments/01_bandit.py) ----------
N_ARMS = 5
SLOT_ANGLES_RAD = np.linspace(-math.pi / 2, math.pi / 2, N_ARMS)
SLOT_LABELS = ['-90', '-45', '0', '+45', '+90']
TRUE_MEANS = np.array([0.2, 0.4, 0.5, 1.0, 0.3])  # hidden from the agent
REWARD_STD  = 0.5   # noise on each observation
BEST_ARM    = int(np.argmax(TRUE_MEANS))  # = arm 3 (+45 deg)

print(f'Best arm index : {BEST_ARM}  ({SLOT_LABELS[BEST_ARM]} deg)')
print(f'True means     : {TRUE_MEANS}')

### 2a. The Bandit Environment

A bandit is the simplest environment imaginable: no state, one method — `pull(arm)`.

In [ ]:
class BanditEnv:
    """5-armed bandit simulating the base_yaw slot choice."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def pull(self, arm: int) -> float:
        """Return a noisy reward for arm (0-indexed)."""
        return float(TRUE_MEANS[arm] + self.rng.normal(0, REWARD_STD))

env = BanditEnv(seed=42)

# Try pulling each arm three times so you can see the noise
print('arm  angle   reward')
for arm in range(N_ARMS):
    rewards = [env.pull(arm) for _ in range(3)]
    print(f'{arm:3d}  {SLOT_LABELS[arm]:6s}  {rewards}')

### 2b. Three Exploration Strategies

**Greedy** — always pick the arm with the highest estimated value.  Fast to
exploit, but it can lock onto a sub-optimal arm and never recover.

**Epsilon-greedy (ε-greedy)** — be greedy most of the time, but with
probability ε pick a random arm.  The classic balance used in DQN.

**Optimistic initial values** — start Q-estimates very high.  The agent is
"disappointed" by every arm it tries and naturally explores all of them before
settling.  No random coin flip needed.

In [ ]:
def run_bandit(strategy='epsilon_greedy', pulls=400, epsilon=0.1, init_q=0.0, seed=0):
    """Run a bandit agent and return per-step reward and regret arrays."""
    rng = np.random.default_rng(seed)
    env = BanditEnv(seed=seed + 1)

    Q = np.full(N_ARMS, init_q, dtype=float)   # estimated Q-values
    N = np.zeros(N_ARMS, dtype=int)             # pull counts

    rewards = np.empty(pulls)
    regrets = np.empty(pulls)

    for t in range(pulls):
        # ---- action selection ----
        if strategy == 'greedy':
            arm = int(np.argmax(Q))
        elif strategy == 'epsilon_greedy':
            if rng.random() < epsilon:
                arm = int(rng.integers(N_ARMS))  # explore
            else:
                arm = int(np.argmax(Q))          # exploit
        else:  # optimistic_init: same as greedy but Q started high
            arm = int(np.argmax(Q))

        # ---- pull and update ----
        r = env.pull(arm)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]   # incremental mean update

        rewards[t] = r
        regrets[t] = TRUE_MEANS[BEST_ARM] - TRUE_MEANS[arm]  # instantaneous regret

    return rewards, np.cumsum(regrets), Q

r_greedy,  reg_greedy,  q_g = run_bandit('greedy',         seed=0)
r_eps,     reg_eps,     q_e = run_bandit('epsilon_greedy', seed=0, epsilon=0.1)
r_opt,     reg_opt,     q_o = run_bandit('greedy',         seed=0, init_q=2.0)

print(f'Final cumulative regret after 400 pulls:')
print(f'  Greedy           : {reg_greedy[-1]:.2f}')
print(f'  ε-greedy (ε=0.1) : {reg_eps[-1]:.2f}')
print(f'  Optimistic init  : {reg_opt[-1]:.2f}')

### 2c. Plot cumulative regret

Regret measures the opportunity cost of not always picking the best arm.
Lower regret = better strategy.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless — swap to 'inline' in JupyterLab
import matplotlib.pyplot as plt

steps = np.arange(1, 401)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(steps, reg_greedy, label='Greedy',           color='#e05a2b', lw=1.5)
ax.plot(steps, reg_eps,    label='ε-greedy (ε=0.1)', color='#2b82e0', lw=1.5)
ax.plot(steps, reg_opt,    label='Optimistic init',  color='#2bc45a', lw=1.5)
ax.set_xlabel('Pull #')
ax.set_ylabel('Cumulative regret')
ax.set_title('Bandit: cumulative regret (lower is better)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('/tmp/bandit_regret.png', dpi=110)
plt.close(fig)
print('Saved to /tmp/bandit_regret.png')

### 2d. Inspect learned Q-values

After 400 pulls, which arm does each strategy think is best?  Compare to the
true means.

In [ ]:
print(f'{"Arm":>4}  {"Angle":>6}  {"TrueMean":>9}  {"Greedy Q":>9}  {"ε-Greedy Q":>10}  {"Optimist Q":>10}')
print('-' * 58)
for i in range(N_ARMS):
    marker = ' <-- BEST' if i == BEST_ARM else ''
    print(f'{i:>4}  {SLOT_LABELS[i]:>6}  {TRUE_MEANS[i]:>9.2f}  {q_g[i]:>9.3f}  {q_e[i]:>10.3f}  {q_o[i]:>10.3f}{marker}')

## 3. Key Concepts Revisited

| Concept | In the bandit | In Buddy Jr reach task |
|---------|--------------|------------------------|
| State | None (stateless) | 17-D obs vector |
| Action | Which arm to pull | 4-D joint velocity jog |
| Reward | Noisy scalar per pull | Distance reduction + success bonus |
| Episode | Single pull | Up to 200 steps |
| Policy | Q-table (5 entries) | Neural network (PPO / SAC) |

### The Q-value update rule

The incremental mean formula used above is the seed of every Q-learning
algorithm:

```
Q(s, a) <- Q(s, a) + alpha * [r + gamma * max Q(s', a') - Q(s, a)]
```

In the bandit `s` is always the same (stateless), `gamma = 0` (no future),
and `alpha = 1/n` (sample average).  Experiments 03 and 05 extend this to
the full 17-D state space.

## 4. Reward Shaping

The Buddy Jr environment uses **dense rewards**: every step the agent gets a
small signal proportional to how much closer it moved.  Compare that to a
**sparse reward** (only +1 on success, 0 otherwise).

Run the cell below to see the difference between the two reward functions on
a single step.

In [ ]:
SUCCESS_TOL = 0.02   # metres — same as the env default

def dense_reward(dist, prev_dist):
    """Reward for closing the gap; +1 bonus on success."""
    shaping = prev_dist - dist          # positive if we got closer
    success = 1.0 if dist < SUCCESS_TOL else 0.0
    return shaping + success

def sparse_reward(dist):
    """Only +1 on success, -1 per step otherwise."""
    return 0.0 if dist < SUCCESS_TOL else -1.0

examples = [(0.10, 0.09), (0.05, 0.03), (0.025, 0.015), (0.015, 0.010)]
print(f'{"prev_dist":>10}  {"dist":>6}  {"dense":>7}  {"sparse":>7}')
print('-' * 38)
for prev, cur in examples:
    print(f'{prev:>10.3f}  {cur:>6.3f}  {dense_reward(cur, prev):>7.3f}  {sparse_reward(cur):>7.3f}')

## 5. What is a Policy?

A **policy** (often written π) maps observations to actions.  The simplest
policy is random — pick any action uniformly.  A trained neural-network policy
picks the action that (it believes) leads to the most total future reward.

In the cell below we show both for one step in the Buddy Jr environment.

In [ ]:
import rl_lab  # registers BuddyJrReach-v0 and friends
import gymnasium as gym

env = gym.make('BuddyJrReach-v0')
obs, info = env.reset(seed=0)

# --- random policy ---
random_action = env.action_space.sample()
obs2, reward, terminated, truncated, info2 = env.step(random_action)

print('Observation dim :', obs.shape)
print('Random action   :', np.round(random_action, 3))
print('Reward          :', round(reward, 4))
print('Distance to goal:', round(info2['distance'], 4), 'm')
print('Success?        :', info2['is_success'])
env.close()

## 6. Next Steps

You now understand the five fundamental RL building blocks: agent, environment,
observation, action, and reward.  You have also seen the explore/exploit
trade-off in action.

Where to go from here:

- **Notebook 02** — inspect the full Buddy Jr observation vector, step through
  the environment manually, and call the forward kinematics.
- **`experiments/03_qlearning.py`** — extend Q-values to a tabular state space.
- **`experiments/05_dqn.py`** — replace the Q-table with a neural network.
- **`experiments/08_ppo.py`** — learn in the continuous action space.